##### Copyright 2026 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Full fine-tuning

[(Q)LoRA](qlora.md) works well for most tasks because it only trains small adapter matrices on top of a frozen model and tends to have a decent accuracy. In contrast, full fine-tuning unfreezes the entire model and updates every parameter in 16-bit precision (bfloat16) which tends to result in higher performance.

You can choose full fine-tuning over (Q)LoRA when you are training on a large dataset, teaching the model a very different output style, or preparing a checkpoint for further alignment. For multi-GPU training on larger checkpoints, see [Scaling](scaling.md).

---

## 1. Full fine-tuning versus LoRA

In LoRA, the base embeddings and decoder layers stay frozen in memory and only a small adapter is trainable. In full fine-tuning, every weight matrix in the model is trainable. Each backward pass computes a full gradient matrix and updates optimizer states:

![LoRA versus full fine-tuning memory and gradient flow](https://gemma.ai/assets/tuning/fft_01_full_backprop_flow.svg)

LoRA does not touch the embeddings and layers during the backward pass. Full fine-tuning, in contrast, allocates gradients and optimizer states for the full model. Since every parameter is updated, full fine-tuning keeps three copies of the model in GPU memory:

- **Weights:** 2 bytes per parameter in bfloat16.
- **Gradients:** 2 bytes per parameter in bfloat16.
- **Optimizer states:** 8 bytes per parameter in float32, or 2 bytes per parameter when using an 8-bit AdamW optimizer.

---

## 2. Warmup and learning rates

The most common mistake when switching from LoRA to full fine-tuning is keeping LoRA's learning rate (2e-4).

In LoRA, the second adapter matrix starts at zero, so early steps make tiny changes to the model's output. In full fine-tuning, the very first step modifies the pretrained weights directly before the optimizer has calibrated its step sizes:

![Learning rate schedule and early gradient size](https://gemma.ai/assets/tuning/fft_02_warmup_cosine_schedule.svg)

Without warmup, the first few batches create a steep gradient spike that can overwrite pretrained attention heads. A 10% linear warmup, a 10x lower learning rate (2e-5), and gradient clipping keep early updates small while the optimizer calibrates its second-moment estimates.

---

## 3. Set up the environment

We will fine-tune Gemma 4 on the [bebechien/MobileGameNPC](https://huggingface.co/datasets/bebechien/MobileGameNPC) dataset. It is a small dataset that contains a alien dialect used for an NPC in a game. This will allow the model to talk in a very different manner.

First, we install the post-training and evaluation libraries:

In [ ]:
# Install Hugging Face post-training, quantization, multimodal, and evaluation libraries
%pip install -q -U "transformers>=5.10.1" "trl>=0.16.0" "peft>=0.19.0" accelerate datasets bitsandbytes torchao
%pip install -q -U evaluate sacrebleu jiwer pillow soundfile librosa matplotlib tensorboard


If you plan to push your trained checkpoint to the Hugging Face Hub, log in with a write token:

In [ ]:
from huggingface_hub import login

# Optional: uncomment to push your trained checkpoint to the Hugging Face Hub
# login()


---

## 4. Prepare the dataset

The [bebechien/MobileGameNPC](https://huggingface.co/datasets/bebechien/MobileGameNPC) dataset has pairs of player questions with replies from the NPC. The player has no dialect but the NPC swaps "s" sounds for "z", says "da" for "the", adds clicks like `*k'tak*`, etc.

We format each row into:

* a `prompt` list (the player's turn)
* a `completion` list (the NPC's reply)

With this prompt-completion format, TRL automatically computes loss only on the NPC's reply, as explained in [Dataset formatting](datasets-formatting.md#2-masking-prompt-tokens):

In [ ]:
from datasets import load_dataset


# Format each row into player prompt and NPC completion turns
def create_conversation(sample):
    return {
        "prompt": [
            {"role": "user", "content": sample["player"]},
        ],
        "completion": [
            {"role": "assistant", "content": sample["alien"]},
        ],
    }


# Load the Martian NPC split and hold out 20% for validation
dataset = load_dataset("bebechien/MobileGameNPC", "martian", split="train")
dataset = dataset.map(create_conversation, remove_columns=dataset.features, batched=False)
dataset = dataset.train_test_split(test_size=0.2, shuffle=False)

print(dataset["train"][0]["prompt"] + dataset["train"][0]["completion"])


---

## 5. Load the model

We select the instruction-tuned Gemma 4 checkpoint and load it in `bfloat16`:

In [ ]:
import torch
from transformers import pipeline

model_id = "google/gemma-4-E4B-it"
checkpoint_dir = "gemma-4-npc-full"
learning_rate = 5e-5

# Load the base model in bfloat16 for baseline comparison
torch.manual_seed(42)
pipe = pipeline("text-generation", model=model_id, dtype=torch.bfloat16, device_map="auto")


---

## 6. Evaluate the base model

Next, we test how the unmodified model responds to a sample question. The sample question is taken from the test split and this version of the model was not trained yet for this particular task:

In [ ]:
# Generate a baseline response on a held-out test prompt
test_sample = dataset["test"][2]
prompt = pipe.tokenizer.apply_chat_template(
    test_sample["prompt"], tokenize=False, add_generation_prompt=True
)
outputs = pipe(prompt, max_new_tokens=128, disable_compile=True)

print(f"Question:\n{test_sample['prompt'][0]['content']}\n")
print(f"Target Dialect:\n{test_sample['completion'][0]['content']}\n")
print(f"Base Model Output:\n{outputs[0]['generated_text'][len(prompt):].strip()}")


Note how the model remains in its own dialect. That is to be expected since we did not give the model instructions to do otherwise.

Next, let's delete the pipeline and empty the VRAM before we can train the model:

In [ ]:
# Free the inference pipeline before starting full fine-tuning
del pipe
torch.cuda.empty_cache()


---

## 7. Train the model

To train the model on this dataset, we create the `SFTConfig` which contains the configuration for the trainer. There are some differences in the parameters compared QLoRA:

1. **`learning_rate = 5e-5`:** Four times lower than QLoRA (`2e-4`).
2. **`lr_scheduler_type = "cosine"` and `warmup_steps = 0.10`:** Ramps the learning rate up over the first 10% of steps.
3. **`max_grad_norm = 1.0`:** Clips sudden gradient spikes.
4. **`optim = "adamw_torch_8bit"`:** Uses TorchAO's 8-bit AdamW to store every parameter's optimizer state in 2 bytes instead of 8 bytes (~28–32 GB peak VRAM on `E2B`).

> **Note:** With `bitsandbytes` (`adamw_bnb_8bit` or `paged_adamw_8bit`), `transformers` keeps every `nn.Embedding` table in 32-bit precision. On Gemma 4 `E2B` and `E4B`, the token and Per-Layer Embedding (PLE) lookup tables hold over half the model's weights, adding ~22 GB of 32-bit optimizer state and causing out-of-memory errors. `adamw_torch_8bit` quantizes optimizer states to 8 bits across every layer, including PLE.

In [ ]:
from trl import SFTConfig, SFTTrainer

# Set the configuration for the trainer
args = SFTConfig(
    output_dir=checkpoint_dir,
    max_length=512,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    optim="adamw_torch_8bit",
    logging_steps=1,
    save_strategy="no",             # Final checkpoint is saved below with trainer.save_model()
    eval_strategy="epoch",
    eval_on_start=True,
    learning_rate=learning_rate,
    warmup_steps=0.10,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=1.0,
    bf16=True,
    model_init_kwargs={"dtype": torch.bfloat16},
    push_to_hub=False,
    report_to="tensorboard",
)

# Define and run the trainer
trainer = SFTTrainer(
    model=model_id,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

trainer.train()
trainer.save_model()


Both the validation and training losses drop sharply over the first two epochs! Notice that validation loss reaches its lowest point at **epoch 2 (`3.1154`)** and creeps back up to `3.4115` by epoch 5 as all 2B weights begin memorizing the small 160-row dataset (`0.0311` training loss). For general domain tasks, it is advised to stop at the epoch where validation loss reaches its lowest point (or set `load_best_model_at_end=True`). When you want a very strict character persona, training a few additional epochs can strengthen style adherence—potentially at the cost of general capabilities.

As always, let's free up the GPU memory before continuing:

In [ ]:
# Free trainer memory before loading the saved checkpoint
del trainer
torch.cuda.empty_cache()


---

## 8. Evaluate the fine-tuned model

To test the fine-tuned model, we reload the saved checkpoint and test its responses on the held-out test split:

In [ ]:
from transformers import pipeline

# Load the saved fine-tuned checkpoint and evaluate on the test split
torch.manual_seed(42)
ft_pipe = pipeline("text-generation", model=checkpoint_dir, dtype=torch.bfloat16, device_map="auto")

for item in dataset["test"]:
    prompt = ft_pipe.tokenizer.apply_chat_template(
        item["prompt"], tokenize=False, add_generation_prompt=True
    )
    outputs = ft_pipe(prompt, max_new_tokens=128, disable_compile=True)
    generated = outputs[0]["generated_text"][len(prompt) :].strip().removesuffix("<turn|>")
    print(f"Player: {item['prompt'][0]['content']}")
    print(f"NPC:    {generated}")
    print("-" * 60)


With a full fine-tune, the model now nicely follows the training data and stays in character!

---